In [41]:
# =============================================================================
# CELL 1 — Imports and environment verification
#
# Kernel: CFB Model (ARM)  ~/miniforge3/envs/cfb_model_arm/bin/python
# NumPyro 0.21.0 / JAX 0.10.0 confirmed working on cpu backend (Day 20).
#
# DB connection opened here and held open for the full notebook.
# Do not close until the final cell.
# =============================================================================

import numpy as np
import pandas as pd
import psycopg2
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from jax import random
from dataclasses import dataclass
from typing import Optional
import time

conn = psycopg2.connect(
    host='127.0.0.1', port=5455, dbname='postgres',
    user='postgres', password='postgres'
)
cur = conn.cursor()

print(f"NumPyro version : {numpyro.__version__}")
print(f"JAX version     : {jax.__version__}")
print(f"JAX backend     : {jax.default_backend()}")
print(f"pandas          : {pd.__version__}")
print(f"numpy           : {np.__version__}")
print()
print("DB connection established.")
print("Cell 1 complete.")

NumPyro version : 0.21.0
JAX version     : 0.10.0
JAX backend     : cpu
pandas          : 3.0.2
numpy           : 2.4.4

DB connection established.
Cell 1 complete.


In [42]:
# =============================================================================
# CELL 2 — Reproduce conference index maps and model infrastructure
#
# Changes from model_02_architecture.ipynb:
#
#   1. r_negbinom prior changed from HalfNormal(5.0) to Gamma(2.0, 0.1).
#      HalfNormal(5.0) puts too much mass near zero; with r→0 the NegBinom2
#      likelihood becomes degenerate (rate = r/mu → 0), producing a near-
#      vertical wall in the log-likelihood that NUTS cannot traverse.
#      Gamma(2.0, 0.1) has mean=20, is strictly positive, and concentrated
#      in the plausible range for CFB scoring (implied r≈5–15 from observed
#      variance ~130–150 with mean ~27).
#
#   2. Non-centered parameterization for alpha_team, delta_team, hfa_team.
#      Centered form: alpha_team ~ Normal(0, sigma_attack) creates Neal's
#      funnel geometry — NUTS must simultaneously adjust sigma and all 131
#      team values. Non-centered form separates them:
#        alpha_team_raw ~ Normal(0, 1)          [sampled]
#        alpha_team = alpha_team_raw * sigma     [deterministic]
#      The log_mu equation is mathematically identical. Only the geometry
#      seen by NUTS changes.
#
#   3. mu and r clamped >= 1e-6, numpyro.enable_validation(False) used in
#      Cell 6 — both retained from previous iteration.
# =============================================================================

# --- Conference index maps ---
CONFERENCES = [
    "ACC",               # 0
    "American Athletic", # 1
    "Big 12",            # 2
    "Big Ten",           # 3
    "Conference USA",    # 4
    "Mid-American",      # 5
    "Mountain West",     # 6
    "Pac-12",            # 7
    "SEC",               # 8
    "Sun Belt",          # 9
]
N_CONFERENCES = len(CONFERENCES)
conf_to_idx   = {c: i for i, c in enumerate(CONFERENCES)}
SUN_BELT_IDX  = conf_to_idx["Sun Belt"]

# --- Conference-scope sets ---
SCOPE_LAST3_OFF_EPA      = {"ACC", "Mid-American", "SEC"}
SCOPE_LAST3_DEF_EPA      = {"American Athletic", "Big Ten", "Conference USA",
                             "Mid-American", "Pac-12", "Sun Belt"}
SCOPE_LAST3_PTS_SCORED   = {"ACC", "Big 12", "Big Ten", "Conference USA",
                             "Mid-American", "Mountain West"}
SCOPE_LAST3_PTS_ALLOWED  = {"American Athletic", "Big Ten", "Conference USA",
                             "Mountain West", "Pac-12", "Sun Belt"}
SCOPE_DAYS_SINCE         = {"American Athletic", "Big 12"}
SCOPE_CLOSE_PLAY_COUNT   = {"ACC", "American Athletic", "Big 12",
                             "Mid-American", "Pac-12", "Sun Belt"}
SCOPE_ELEVATION          = {"Mountain West", "Big 12"}


def build_conf_mask(team_conferences, scope_set):
    return jnp.array(
        [1.0 if c in scope_set else 0.0 for c in team_conferences],
        dtype=jnp.float32
    )


# --- GameData dataclass ---
@dataclass
class GameData:
    team_idx  : jnp.ndarray
    opp_idx   : jnp.ndarray
    conf_idx  : jnp.ndarray
    is_home   : jnp.ndarray
    points    : Optional[jnp.ndarray]

    sp_rating          : jnp.ndarray
    recruiting_3yr_avg : jnp.ndarray

    close_game_epa        : jnp.ndarray
    close_game_def_epa    : jnp.ndarray
    pregame_elo           : jnp.ndarray
    elo_sp_divergence     : jnp.ndarray
    last3_win_pct         : jnp.ndarray
    away_travel_distance  : jnp.ndarray
    away_tz_delta         : jnp.ndarray
    wind_chill            : jnp.ndarray
    rush_rate_std_downs   : jnp.ndarray
    rush_rate_pass_downs  : jnp.ndarray
    off_sack_rate_allowed : jnp.ndarray
    def_sack_rate         : jnp.ndarray
    offense_archetype     : jnp.ndarray
    defense_archetype     : jnp.ndarray
    home_off_away_def     : jnp.ndarray
    away_off_home_def     : jnp.ndarray

    last3_off_epa_avg      : jnp.ndarray
    mask_last3_off_epa     : jnp.ndarray
    last3_def_epa_avg      : jnp.ndarray
    mask_last3_def_epa     : jnp.ndarray
    last3_pts_scored_avg   : jnp.ndarray
    mask_last3_pts_scored  : jnp.ndarray
    last3_pts_allowed_avg  : jnp.ndarray
    mask_last3_pts_allowed : jnp.ndarray
    days_since_last_game   : jnp.ndarray
    mask_days_since        : jnp.ndarray
    close_play_count_delta : jnp.ndarray
    mask_close_play_count  : jnp.ndarray
    away_elevation_delta   : jnp.ndarray
    mask_elevation         : jnp.ndarray


# --- model_cfb() — non-centered parameterization ---
def model_cfb(data: GameData, N_teams: int):

    # =========================================================================
    # LEAGUE LEVEL
    # =========================================================================
    mu_league  = numpyro.sample("mu_league",  dist.Normal(3.3, 0.2))
    hfa_league = numpyro.sample("hfa_league", dist.Normal(0.1, 0.05))

    # Gamma(2.0, 0.1): mean=20, strictly positive, concentrated away from
    # zero. Replaces HalfNormal(5.0) which collapsed to ~0 in first fit.
    r_negbinom = numpyro.sample("r_negbinom", dist.Gamma(2.0, 0.1))

    # =========================================================================
    # CONFERENCE LEVEL — centered (10 groups, no funnel risk)
    # =========================================================================
    sigma_conference = numpyro.sample("sigma_conference", dist.HalfNormal(3.0))
    mu_conference    = numpyro.sample(
        "mu_conference",
        dist.Normal(0.0, sigma_conference),
        sample_shape=(N_CONFERENCES,)
    )

    # =========================================================================
    # TEAM LEVEL — non-centered parameterization
    # Sampled: *_raw ~ Normal(0, 1)
    # Deterministic: * = *_raw * sigma
    # Identical math to centered form; NUTS sees unit-normal geometry instead
    # of funnel geometry.
    # =========================================================================

    # Attack
    sigma_attack    = numpyro.sample("sigma_attack", dist.HalfNormal(0.4))
    alpha_team_raw  = numpyro.sample(
        "alpha_team_raw",
        dist.Normal(0.0, 1.0),
        sample_shape=(N_teams,)
    )
    alpha_team = numpyro.deterministic("alpha_team", alpha_team_raw * sigma_attack)

    # Defense
    sigma_defense   = numpyro.sample("sigma_defense", dist.HalfNormal(0.4))
    delta_team_raw  = numpyro.sample(
        "delta_team_raw",
        dist.Normal(0.0, 1.0),
        sample_shape=(N_teams,)
    )
    delta_team = numpyro.deterministic("delta_team", delta_team_raw * sigma_defense)

    # Team HFA deviation
    sigma_hfa_team  = numpyro.sample("sigma_hfa_team", dist.HalfNormal(0.1))
    hfa_team_raw    = numpyro.sample(
        "hfa_team_raw",
        dist.Normal(0.0, 1.0),
        sample_shape=(N_teams,)
    )
    hfa_team = numpyro.deterministic("hfa_team", hfa_team_raw * sigma_hfa_team)

    # =========================================================================
    # PRIOR SEEDS — team level
    # =========================================================================
    sp_weight = numpyro.sample("sp_weight", dist.Normal(0.0, 1.0))

    rec_weight_other   = numpyro.sample(
        "rec_weight_other",
        dist.Normal(0.0, 0.5),
        sample_shape=(N_CONFERENCES - 1,)
    )
    rec_weight_sunbelt = numpyro.sample(
        "rec_weight_sunbelt",
        dist.TruncatedNormal(0.0, 0.5, high=0.0)
    )
    rec_weight = jnp.concatenate([
        rec_weight_other[:SUN_BELT_IDX],
        rec_weight_sunbelt[jnp.newaxis],
        rec_weight_other[SUN_BELT_IDX:]
    ])

    # =========================================================================
    # GAME-LEVEL FEATURE COEFFICIENTS
    # =========================================================================
    b_close_game_epa         = numpyro.sample("b_close_game_epa",         dist.Normal(0.0, 0.5))
    b_close_game_def_epa     = numpyro.sample("b_close_game_def_epa",     dist.Normal(0.0, 0.5))
    b_pregame_elo            = numpyro.sample("b_pregame_elo",            dist.Normal(0.0, 0.3))
    b_elo_sp_divergence      = numpyro.sample("b_elo_sp_divergence",      dist.Normal(0.0, 0.2))
    b_last3_win_pct          = numpyro.sample("b_last3_win_pct",          dist.Normal(0.0, 0.3))
    b_away_travel_distance   = numpyro.sample("b_away_travel_distance",   dist.Normal(0.0, 0.3))
    b_away_tz_delta          = numpyro.sample("b_away_tz_delta",          dist.Normal(0.0, 0.3))
    b_wind_chill             = numpyro.sample("b_wind_chill",             dist.Normal(0.0, 0.2))
    b_rush_rate_std_downs    = numpyro.sample("b_rush_rate_std_downs",    dist.Normal(0.0, 0.3))
    b_rush_rate_pass_downs   = numpyro.sample("b_rush_rate_pass_downs",   dist.Normal(0.0, 0.3))
    b_off_sack_rate_allowed  = numpyro.sample("b_off_sack_rate_allowed",  dist.Normal(0.0, 0.2))
    b_def_sack_rate          = numpyro.sample("b_def_sack_rate",          dist.Normal(0.0, 0.2))
    b_offense_archetype      = numpyro.sample("b_offense_archetype",      dist.Normal(0.0, 0.3))
    b_defense_archetype      = numpyro.sample("b_defense_archetype",      dist.Normal(0.0, 0.3))
    b_home_off_away_def      = numpyro.sample("b_home_off_away_def",      dist.Normal(0.0, 0.3))
    b_away_off_home_def      = numpyro.sample("b_away_off_home_def",      dist.Normal(0.0, 0.3))
    b_last3_off_epa_avg      = numpyro.sample("b_last3_off_epa_avg",      dist.Normal(0.0, 0.3))
    b_last3_def_epa_avg      = numpyro.sample("b_last3_def_epa_avg",      dist.Normal(0.0, 0.3))
    b_last3_pts_scored_avg   = numpyro.sample("b_last3_pts_scored_avg",   dist.Normal(0.0, 0.3))
    b_last3_pts_allowed_avg  = numpyro.sample("b_last3_pts_allowed_avg",  dist.Normal(0.0, 0.3))
    b_days_since_last_game   = numpyro.sample("b_days_since_last_game",   dist.Normal(0.0, 0.2))
    b_close_play_count_delta = numpyro.sample("b_close_play_count_delta", dist.Normal(0.0, 0.2))
    b_away_elevation_delta   = numpyro.sample("b_away_elevation_delta",   dist.Normal(0.0, 0.3))

    # =========================================================================
    # LINEAR PREDICTOR
    # =========================================================================
    log_mu = (
        mu_league
        + mu_conference[data.conf_idx]
        + alpha_team[data.team_idx]
        - delta_team[data.opp_idx]
        + (hfa_league + hfa_team[data.team_idx]) * data.is_home
        + sp_weight * data.sp_rating
        + rec_weight[data.conf_idx] * data.recruiting_3yr_avg
        + b_close_game_epa     * data.close_game_epa
        + b_close_game_def_epa * data.close_game_def_epa
        + b_pregame_elo        * data.pregame_elo
        + b_elo_sp_divergence  * data.elo_sp_divergence
        + b_last3_win_pct      * data.last3_win_pct
        + b_away_travel_distance * data.away_travel_distance
        + b_away_tz_delta        * data.away_tz_delta
        + b_wind_chill           * data.wind_chill
        + b_rush_rate_std_downs  * data.rush_rate_std_downs
        + b_rush_rate_pass_downs * data.rush_rate_pass_downs
        + b_off_sack_rate_allowed * data.off_sack_rate_allowed
        + b_def_sack_rate         * data.def_sack_rate
        + b_offense_archetype * data.offense_archetype
        + b_defense_archetype * data.defense_archetype
        + b_home_off_away_def * data.home_off_away_def
        + b_away_off_home_def * data.away_off_home_def
        + b_last3_off_epa_avg    * data.mask_last3_off_epa    * data.last3_off_epa_avg
        + b_last3_def_epa_avg    * data.mask_last3_def_epa    * data.last3_def_epa_avg
        + b_last3_pts_scored_avg * data.mask_last3_pts_scored * data.last3_pts_scored_avg
        + b_last3_pts_allowed_avg * data.mask_last3_pts_allowed * data.last3_pts_allowed_avg
        + b_days_since_last_game * data.mask_days_since        * data.days_since_last_game
        + b_close_play_count_delta * data.mask_close_play_count * data.close_play_count_delta
        + b_away_elevation_delta * data.mask_elevation         * data.away_elevation_delta
    )

    mu = jnp.exp(log_mu).clip(1e-6)
    r  = r_negbinom.clip(1e-6)

    numpyro.sample(
        "obs",
        dist.NegativeBinomial2(mu, r, validate_args=False),
        obs=data.points
    )


print("Conference index map:")
for name, idx in conf_to_idx.items():
    tag = "  <- Sun Belt (rec_weight non-positive)" if idx == SUN_BELT_IDX else ""
    print(f"  {idx:2d} : {name}{tag}")
print(f"\nN_CONFERENCES : {N_CONFERENCES}")
print(f"SUN_BELT_IDX  : {SUN_BELT_IDX}")
print()
print("build_conf_mask() defined.")
print("GameData dataclass defined.")
print("model_cfb() defined.")
print("  r_negbinom prior : Gamma(2.0, 0.1)  [mean=20, strictly positive]")
print("  team effects     : non-centered parameterization")
print("Cell 2 complete.")

Conference index map:
   0 : ACC
   1 : American Athletic
   2 : Big 12
   3 : Big Ten
   4 : Conference USA
   5 : Mid-American
   6 : Mountain West
   7 : Pac-12
   8 : SEC
   9 : Sun Belt  <- Sun Belt (rec_weight non-positive)

N_CONFERENCES : 10
SUN_BELT_IDX  : 9

build_conf_mask() defined.
GameData dataclass defined.
model_cfb() defined.
  r_negbinom prior : Gamma(2.0, 0.1)  [mean=20, strictly positive]
  team effects     : non-centered parameterization
Cell 2 complete.


In [43]:
# =============================================================================
# CELL 3 — Load training data from int.int_game_model_features
#
# Season filter: IN (2022, 2023, 2024). 2025 is holdout — never touched.
# FBS integrity check mandatory after load.
# All Decimal columns cast to float64 immediately.
# =============================================================================

load_sql = """
    SELECT *
    FROM int.int_game_model_features
    WHERE season IN (2022, 2023, 2024)
    ORDER BY season, game_id, team_name
"""

cur.execute(load_sql)
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
df = pd.DataFrame(rows, columns=cols)

numeric_cols = [c for c in df.columns if c not in
                ['team_name', 'opponent', 'conference']]
df[numeric_cols] = df[numeric_cols].astype(float)

# --- FBS integrity check ---
print("Conference distribution:")
print(df['conference'].value_counts().sort_index().to_string())
print()

assert 'FBS Independents' not in df['conference'].values, \
    "FBS Independents found in conference — fix before proceeding"
assert 2025 not in df['season'].values, \
    "2025 holdout found in training data — fix before proceeding"
assert df['game_id'].isna().sum() == 0, \
    "Null game_id found"

# --- Shape and basic sanity ---
print(f"Rows loaded        : {len(df):,}")
print(f"Columns            : {len(df.columns)}")
print(f"Unique games       : {df['game_id'].nunique():,}")
print(f"Unique teams       : {df['team_name'].nunique():,}")
print(f"Seasons            : {sorted(df['season'].unique().tolist())}")
print(f"Nulls anywhere     : {df.isna().sum().sum()}")
print(f"points_scored mean : {df['points_scored'].mean():.1f}")
print()

assert len(df) == 3214,          f"Expected 3,214 rows, got {len(df):,}"
assert df['game_id'].nunique() == 1607, \
    f"Expected 1,607 games, got {df['game_id'].nunique():,}"
assert df.isna().sum().sum() == 0, "Nulls found in training data"

print("All load checks passed.")
print("Cell 3 complete.")

Conference distribution:
conference
ACC                  364
American Athletic    320
Big 12               366
Big Ten              420
Conference USA       246
Mid-American         294
Mountain West        282
Pac-12               222
SEC                  358
Sun Belt             342

Rows loaded        : 3,214
Columns            : 33
Unique games       : 1,607
Unique teams       : 131
Seasons            : [2022.0, 2023.0, 2024.0]
Nulls anywhere     : 0
points_scored mean : 26.9

All load checks passed.
Cell 3 complete.


In [44]:
# =============================================================================
# CELL 4 — Build index arrays and conference masks
#
# team_idx and opp_idx: 0-based integer index over unique team_name values.
# Uniqueness key: team_name (team×season pooled — 131 unique teams across
# 2022-2024; no separate index per season per locked decision).
#
# conf_idx: maps each row's conference string to conf_to_idx integer.
# Must use the exact same conf_to_idx from Cell 2.
#
# opp_idx: derived from the opponent column in int_game_model_features,
# which carries the opponent team_name for each row.
#
# All index arrays must satisfy:
#   team_idx in [0, N_teams)
#   opp_idx  in [0, N_teams)
#   conf_idx in [0, N_CONFERENCES)
# =============================================================================

# --- Team index map ---
unique_teams = sorted(df['team_name'].unique())
N_teams      = len(unique_teams)
team_to_idx  = {name: i for i, name in enumerate(unique_teams)}

print(f"N_teams : {N_teams}")

# --- Verify all opponents are in the team map ---
unknown_opps = set(df['opponent'].unique()) - set(team_to_idx.keys())
assert len(unknown_opps) == 0, \
    f"Opponents not in team_to_idx: {unknown_opps}"

# --- Build index arrays ---
team_idx = jnp.array(df['team_name'].map(team_to_idx).values, dtype=jnp.int32)
opp_idx  = jnp.array(df['opponent'].map(team_to_idx).values,  dtype=jnp.int32)
conf_idx = jnp.array(df['conference'].map(conf_to_idx).values, dtype=jnp.int32)

# --- Verify conference mapping was complete ---
assert df['conference'].map(conf_to_idx).isna().sum() == 0, \
    f"Unmapped conferences: {set(df['conference'].unique()) - set(conf_to_idx.keys())}"

# --- Range assertions ---
assert int(team_idx.min()) >= 0 and int(team_idx.max()) < N_teams, \
    f"team_idx out of range: [{int(team_idx.min())}, {int(team_idx.max())}]"
assert int(opp_idx.min()) >= 0 and int(opp_idx.max()) < N_teams, \
    f"opp_idx out of range: [{int(opp_idx.min())}, {int(opp_idx.max())}]"
assert int(conf_idx.min()) >= 0 and int(conf_idx.max()) < N_CONFERENCES, \
    f"conf_idx out of range: [{int(conf_idx.min())}, {int(conf_idx.max())}]"

print(f"team_idx : shape={team_idx.shape}  min={int(team_idx.min())}  max={int(team_idx.max())}")
print(f"opp_idx  : shape={opp_idx.shape}  min={int(opp_idx.min())}  max={int(opp_idx.max())}")
print(f"conf_idx : shape={conf_idx.shape}  min={int(conf_idx.min())}  max={int(conf_idx.max())}")

# --- Build conference masks ---
team_conferences = df['conference'].tolist()

mask_last3_off_epa     = build_conf_mask(team_conferences, SCOPE_LAST3_OFF_EPA)
mask_last3_def_epa     = build_conf_mask(team_conferences, SCOPE_LAST3_DEF_EPA)
mask_last3_pts_scored  = build_conf_mask(team_conferences, SCOPE_LAST3_PTS_SCORED)
mask_last3_pts_allowed = build_conf_mask(team_conferences, SCOPE_LAST3_PTS_ALLOWED)
mask_days_since        = build_conf_mask(team_conferences, SCOPE_DAYS_SINCE)
mask_close_play_count  = build_conf_mask(team_conferences, SCOPE_CLOSE_PLAY_COUNT)
mask_elevation         = build_conf_mask(team_conferences, SCOPE_ELEVATION)

print()
print("Conference mask non-zero counts (of 3,214 rows):")
print(f"  mask_last3_off_epa     : {int(mask_last3_off_epa.sum()):,}")
print(f"  mask_last3_def_epa     : {int(mask_last3_def_epa.sum()):,}")
print(f"  mask_last3_pts_scored  : {int(mask_last3_pts_scored.sum()):,}")
print(f"  mask_last3_pts_allowed : {int(mask_last3_pts_allowed.sum()):,}")
print(f"  mask_days_since        : {int(mask_days_since.sum()):,}")
print(f"  mask_close_play_count  : {int(mask_close_play_count.sum()):,}")
print(f"  mask_elevation         : {int(mask_elevation.sum()):,}")

# Mask totals must match sum of in-scope conference row counts from Cell 3
assert int(mask_last3_off_epa.sum()) == 364 + 294 + 358, \
    f"mask_last3_off_epa sum wrong: {int(mask_last3_off_epa.sum())}"

print()
print("All index and mask checks passed.")
print("Cell 4 complete.")

N_teams : 131
team_idx : shape=(3214,)  min=0  max=130
opp_idx  : shape=(3214,)  min=0  max=130
conf_idx : shape=(3214,)  min=0  max=9

Conference mask non-zero counts (of 3,214 rows):
  mask_last3_off_epa     : 1,016
  mask_last3_def_epa     : 1,844
  mask_last3_pts_scored  : 1,972
  mask_last3_pts_allowed : 1,832
  mask_days_since        : 686
  mask_close_play_count  : 1,908
  mask_elevation         : 648

All index and mask checks passed.
Cell 4 complete.


In [45]:
# =============================================================================
# CELL 5 — Construct GameData
#
# All continuous features standardized to mean=0, std=1 before model input.
# Standardizing only 3 features was an oversight — NUTS requires all
# continuous inputs on comparable scales for efficient geometry, and the
# Normal(0, 0.2-0.5) coefficient priors only make sense if features are
# on a unit scale.
#
# Excluded from standardization:
#   - is_home          : binary {0, 1}
#   - points_scored    : observed outcome, not a predictor
#   - offense/defense/home_off/away_off archetype matchup : integer category
#     codes, not a continuous magnitude
#   - conference masks : binary {0, 1}
#
# Stats computed on training data only (2022-2024).
# int.int_game_model_features is not modified.
# Standardization stats printed for audit trail.
# =============================================================================

CONTINUOUS_FEATURES = [
    'sp_rating',
    'recruiting_3yr_avg',
    'close_game_epa_per_play',
    'close_game_def_epa_per_play',
    'pregame_elo',
    'elo_sp_divergence',
    'last3_win_pct',
    'away_travel_distance_mi',
    'away_tz_delta_hrs',
    'wind_chill',
    'rush_rate_std_downs_delta',
    'rush_rate_pass_downs_delta',
    'off_sack_rate_allowed_delta',
    'def_sack_rate_delta',
    'last3_off_epa_avg',
    'last3_def_epa_avg',
    'last3_points_scored_avg',
    'last3_points_allowed_avg',
    'days_since_last_game',
    'close_game_play_count_delta',
    'away_elevation_delta_ft',
]

# Store standardization stats for use at prediction time
scaler_stats = {}
print("Standardizing continuous features:")
for col in CONTINUOUS_FEATURES:
    mean = df[col].mean()
    std  = df[col].std()
    # If std is zero (constant column) leave as-is to avoid divide-by-zero
    if std == 0:
        print(f"  WARNING: {col} has std=0 — skipping standardization")
        scaler_stats[col] = {'mean': float(mean), 'std': 1.0}
    else:
        df[col] = (df[col] - mean) / std
        scaler_stats[col] = {'mean': float(mean), 'std': float(std)}
        print(f"  {col:<35s} mean={df[col].mean():+.4f}  std={df[col].std():.4f}  "
              f"(raw mean={scaler_stats[col]['mean']:.3f}, std={scaler_stats[col]['std']:.3f})")

print()

def to_f32(col):
    return jnp.array(df[col].values, dtype=jnp.float32)

training_data = GameData(
    team_idx  = team_idx,
    opp_idx   = opp_idx,
    conf_idx  = conf_idx,

    is_home   = jnp.array(df['is_home'].values, dtype=jnp.float32),
    points    = jnp.array(df['points_scored'].values, dtype=jnp.int32),

    sp_rating          = to_f32('sp_rating'),
    recruiting_3yr_avg = to_f32('recruiting_3yr_avg'),

    close_game_epa        = to_f32('close_game_epa_per_play'),
    close_game_def_epa    = to_f32('close_game_def_epa_per_play'),
    pregame_elo           = to_f32('pregame_elo'),
    elo_sp_divergence     = to_f32('elo_sp_divergence'),
    last3_win_pct         = to_f32('last3_win_pct'),
    away_travel_distance  = to_f32('away_travel_distance_mi'),
    away_tz_delta         = to_f32('away_tz_delta_hrs'),
    wind_chill            = to_f32('wind_chill'),
    rush_rate_std_downs   = to_f32('rush_rate_std_downs_delta'),
    rush_rate_pass_downs  = to_f32('rush_rate_pass_downs_delta'),
    off_sack_rate_allowed = to_f32('off_sack_rate_allowed_delta'),
    def_sack_rate         = to_f32('def_sack_rate_delta'),
    offense_archetype     = to_f32('offense_archetype_matchup'),
    defense_archetype     = to_f32('defense_archetype_matchup'),
    home_off_away_def     = to_f32('home_off_vs_away_def_matchup'),
    away_off_home_def     = to_f32('away_off_vs_home_def_matchup'),

    last3_off_epa_avg      = to_f32('last3_off_epa_avg'),
    mask_last3_off_epa     = mask_last3_off_epa,
    last3_def_epa_avg      = to_f32('last3_def_epa_avg'),
    mask_last3_def_epa     = mask_last3_def_epa,
    last3_pts_scored_avg   = to_f32('last3_points_scored_avg'),
    mask_last3_pts_scored  = mask_last3_pts_scored,
    last3_pts_allowed_avg  = to_f32('last3_points_allowed_avg'),
    mask_last3_pts_allowed = mask_last3_pts_allowed,
    days_since_last_game   = to_f32('days_since_last_game'),
    mask_days_since        = mask_days_since,
    close_play_count_delta = to_f32('close_game_play_count_delta'),
    mask_close_play_count  = mask_close_play_count,
    away_elevation_delta   = to_f32('away_elevation_delta_ft'),
    mask_elevation         = mask_elevation,
)

# --- Verification ---
print("GameData field shapes and dtypes:")
print(f"  team_idx               : {training_data.team_idx.shape}  {training_data.team_idx.dtype}")
print(f"  opp_idx                : {training_data.opp_idx.shape}  {training_data.opp_idx.dtype}")
print(f"  conf_idx               : {training_data.conf_idx.shape}  {training_data.conf_idx.dtype}")
print(f"  is_home                : {training_data.is_home.shape}  {training_data.is_home.dtype}")
print(f"  points                 : {training_data.points.shape}  {training_data.points.dtype}")
print(f"  sp_rating              : {training_data.sp_rating.shape}  {training_data.sp_rating.dtype}")
print(f"  close_game_epa         : {training_data.close_game_epa.shape}  {training_data.close_game_epa.dtype}")
print(f"  mask_last3_off_epa     : {training_data.mask_last3_off_epa.shape}  {training_data.mask_last3_off_epa.dtype}")
print()

N_rows = len(df)
import dataclasses
for field in dataclasses.fields(training_data):
    val = getattr(training_data, field.name)
    if val is not None:
        assert val.shape[0] == N_rows, \
            f"{field.name} shape[0] = {val.shape[0]}, expected {N_rows}"

print(f"All {len(dataclasses.fields(training_data))} GameData fields verified: shape[0] == {N_rows}.")
print()

# Verify all continuous features are now on unit scale
print("Post-standardization scale check:")
for col in CONTINUOUS_FEATURES:
    mean = df[col].mean()
    std  = df[col].std()
    assert abs(mean) < 1e-6, f"{col} mean not zero: {mean:.6f}"
    assert abs(std - 1.0) < 1e-6, f"{col} std not 1: {std:.6f}"
print("  All continuous features confirmed mean=0, std=1.")
print()
print("Cell 5 complete.")

Standardized sp_rating                 mean=0.0000  std=1.0000
Standardized recruiting_3yr_avg        mean=-0.0000  std=1.0000
Standardized pregame_elo               mean=-0.0000  std=1.0000

GameData field shapes and dtypes:
  team_idx               : (3214,)  int32
  opp_idx                : (3214,)  int32
  conf_idx               : (3214,)  int32
  is_home                : (3214,)  float32
  points                 : (3214,)  int32
  sp_rating              : (3214,)  float32
  close_game_epa         : (3214,)  float32
  mask_last3_off_epa     : (3214,)  float32

All 37 GameData fields verified: shape[0] == 3214.

Cell 5 complete.


In [46]:
# =============================================================================
# CELL 6 — Run NUTS sampler (diagnostic run)
#
# Uses init_to_value to start chains near the posterior mode identified
# by the log_density diagnostic. Prevents NUTS from initializing in the
# steep tails of the sp_weight / sp_rating likelihood surface, which was
# forcing step size to ~1e-7 and 1,000+ leapfrog steps per sample.
#
# Still 1 chain / 200 warmup / 200 samples until geometry is confirmed
# healthy, then will restore full 4-chain run.
# =============================================================================

from numpyro.infer import init_to_value

init_values = {
    "mu_league"         : jnp.array(3.3),
    "hfa_league"        : jnp.array(0.1),
    "r_negbinom"        : jnp.array(5.0),
    "sigma_conference"  : jnp.array(0.2),
    "mu_conference"     : jnp.zeros(N_CONFERENCES),
    "sigma_attack"      : jnp.array(0.2),
    "alpha_team_raw"    : jnp.zeros(N_teams),
    "sigma_defense"     : jnp.array(0.2),
    "delta_team_raw"    : jnp.zeros(N_teams),
    "sigma_hfa_team"    : jnp.array(0.05),
    "hfa_team_raw"      : jnp.zeros(N_teams),
    "sp_weight"         : jnp.array(0.0),
    "rec_weight_other"  : jnp.zeros(N_CONFERENCES - 1),
    "rec_weight_sunbelt": jnp.array(-0.1),
    "b_close_game_epa"        : jnp.array(0.0),
    "b_close_game_def_epa"    : jnp.array(0.0),
    "b_pregame_elo"           : jnp.array(0.0),
    "b_elo_sp_divergence"     : jnp.array(0.0),
    "b_last3_win_pct"         : jnp.array(0.0),
    "b_away_travel_distance"  : jnp.array(0.0),
    "b_away_tz_delta"         : jnp.array(0.0),
    "b_wind_chill"            : jnp.array(0.0),
    "b_rush_rate_std_downs"   : jnp.array(0.0),
    "b_rush_rate_pass_downs"  : jnp.array(0.0),
    "b_off_sack_rate_allowed" : jnp.array(0.0),
    "b_def_sack_rate"         : jnp.array(0.0),
    "b_offense_archetype"     : jnp.array(0.0),
    "b_defense_archetype"     : jnp.array(0.0),
    "b_home_off_away_def"     : jnp.array(0.0),
    "b_away_off_home_def"     : jnp.array(0.0),
    "b_last3_off_epa_avg"     : jnp.array(0.0),
    "b_last3_def_epa_avg"     : jnp.array(0.0),
    "b_last3_pts_scored_avg"  : jnp.array(0.0),
    "b_last3_pts_allowed_avg" : jnp.array(0.0),
    "b_days_since_last_game"  : jnp.array(0.0),
    "b_close_play_count_delta": jnp.array(0.0),
    "b_away_elevation_delta"  : jnp.array(0.0),
}

nuts_kernel = NUTS(
    model_cfb,
    target_accept_prob=0.9,
    init_strategy=init_to_value(values=init_values),
)

mcmc = MCMC(
    nuts_kernel,
    num_warmup=200,
    num_samples=200,
    num_chains=1,
    progress_bar=True,
)

print("Starting NUTS sampler (diagnostic — 1 chain, 200 warmup, 200 samples)...")
print(f"  N_teams : {N_teams}")
print(f"  N_games : {len(df):,}")
print(f"  init    : init_to_value at prior means / log_density mode")
print()

numpyro.enable_validation(False)
try:
    t0 = time.time()
    mcmc.run(random.PRNGKey(42), data=training_data, N_teams=N_teams)
    fit_time = time.time() - t0
finally:
    numpyro.enable_validation(True)

print(f"\nFit complete. Wall-clock time : {fit_time:.1f}s")

extra = mcmc.get_extra_fields()
n_divergences = int(extra['diverging'].sum()) if 'diverging' in extra else 0
print(f"Divergences : {n_divergences} / 200")

samples = mcmc.get_samples()
r_vals = samples['r_negbinom']
print(f"\nr_negbinom  mean={float(r_vals.mean()):.4f}  "
      f"min={float(r_vals.min()):.4f}  max={float(r_vals.max()):.4f}")
print(f"mu_league   mean={float(samples['mu_league'].mean()):.4f}")
print(f"sp_weight   mean={float(samples['sp_weight'].mean()):.4f}")
print(f"sigma_attack mean={float(samples['sigma_attack'].mean()):.4f}")
print(f"Steps/sample (check progress bar) — target < 64")
print()
print("Cell 6 complete.")

Starting NUTS sampler (diagnostic — 1 chain, 200 warmup, 200 samples)...
  N_teams : 131
  N_games : 3,214
  init    : init_to_value at prior means / log_density mode



sample: 100%|█████████████████████████████████████████████████████████████████| 400/400 [03:24<00:00,  1.96it/s, 1023 steps of size 8.29e-04. acc. prob=0.95]


Fit complete. Wall-clock time : 207.4s
Divergences : 0 / 200

r_negbinom  mean=17.8145  min=15.8421  max=19.9151
mu_league   mean=2.8837
sp_weight   mean=0.1749
sigma_attack mean=0.2228
Steps/sample (check progress bar) — target < 64

Cell 6 complete.


In [47]:
# Diagnostic — do not add to notebook yet
import numpy as np

pts = df['points_scored'].values.astype(float)
print(f"points_scored mean     : {pts.mean():.2f}")
print(f"points_scored variance : {pts.var():.2f}")
print(f"points_scored std      : {pts.std():.2f}")
print(f"implied r (method of moments): {pts.mean()**2 / (pts.var() - pts.mean()):.4f}")
print()
print(f"sp_rating mean  : {df['sp_rating'].mean():.4f}")
print(f"sp_rating std   : {df['sp_rating'].std():.4f}")
print(f"recruiting mean : {df['recruiting_3yr_avg'].mean():.4f}")
print(f"recruiting std  : {df['recruiting_3yr_avg'].std():.4f}")
print(f"pregame_elo mean: {df['pregame_elo'].mean():.4f}")
print(f"pregame_elo std : {df['pregame_elo'].std():.4f}")

points_scored mean     : 26.92
points_scored variance : 165.23
points_scored std      : 12.85
implied r (method of moments): 5.2381

sp_rating mean  : 0.0000
sp_rating std   : 1.0000
recruiting mean : -0.0000
recruiting std  : 1.0000
pregame_elo mean: -0.0000
pregame_elo std : 1.0000


In [48]:
# Diagnostic — prior predictive geometry check
# Do not add to notebook yet.
#
# Evaluates log_prob at the prior mean and at several perturbations to
# understand whether the likelihood surface is well-behaved near reasonable
# parameter values. Uses numpyro.infer.util.log_density directly.

from numpyro.infer.util import log_density

# Build a parameter dict at reasonable starting values
# (prior means / known good values for CFB scoring)
N_t = N_teams

init_params = {
    "mu_league"         : jnp.array(3.3),
    "hfa_league"        : jnp.array(0.1),
    "r_negbinom"        : jnp.array(5.0),
    "sigma_conference"  : jnp.array(0.2),
    "mu_conference"     : jnp.zeros(N_CONFERENCES),
    "sigma_attack"      : jnp.array(0.2),
    "alpha_team_raw"    : jnp.zeros(N_t),
    "sigma_defense"     : jnp.array(0.2),
    "delta_team_raw"    : jnp.zeros(N_t),
    "sigma_hfa_team"    : jnp.array(0.05),
    "hfa_team_raw"      : jnp.zeros(N_t),
    "sp_weight"         : jnp.array(0.0),
    "rec_weight_other"  : jnp.zeros(N_CONFERENCES - 1),
    "rec_weight_sunbelt": jnp.array(-0.1),
    "b_close_game_epa"        : jnp.array(0.0),
    "b_close_game_def_epa"    : jnp.array(0.0),
    "b_pregame_elo"           : jnp.array(0.0),
    "b_elo_sp_divergence"     : jnp.array(0.0),
    "b_last3_win_pct"         : jnp.array(0.0),
    "b_away_travel_distance"  : jnp.array(0.0),
    "b_away_tz_delta"         : jnp.array(0.0),
    "b_wind_chill"            : jnp.array(0.0),
    "b_rush_rate_std_downs"   : jnp.array(0.0),
    "b_rush_rate_pass_downs"  : jnp.array(0.0),
    "b_off_sack_rate_allowed" : jnp.array(0.0),
    "b_def_sack_rate"         : jnp.array(0.0),
    "b_offense_archetype"     : jnp.array(0.0),
    "b_defense_archetype"     : jnp.array(0.0),
    "b_home_off_away_def"     : jnp.array(0.0),
    "b_away_off_home_def"     : jnp.array(0.0),
    "b_last3_off_epa_avg"     : jnp.array(0.0),
    "b_last3_def_epa_avg"     : jnp.array(0.0),
    "b_last3_pts_scored_avg"  : jnp.array(0.0),
    "b_last3_pts_allowed_avg" : jnp.array(0.0),
    "b_days_since_last_game"  : jnp.array(0.0),
    "b_close_play_count_delta": jnp.array(0.0),
    "b_away_elevation_delta"  : jnp.array(0.0),
}

numpyro.enable_validation(False)

ld_base, _ = log_density(
    model_cfb, (training_data,), {"N_teams": N_teams}, init_params
)
print(f"log_density at prior means (all betas=0, r=5) : {ld_base:.2f}")

# Sweep r_negbinom across plausible range
print("\nr_negbinom sweep (all else at prior means):")
for r_val in [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0]:
    p = {**init_params, "r_negbinom": jnp.array(float(r_val))}
    ld, _ = log_density(model_cfb, (training_data,), {"N_teams": N_teams}, p)
    print(f"  r={r_val:<6.2f}  log_density={ld:.2f}")

# Sweep mu_league
print("\nmu_league sweep (r=5, all betas=0):")
for mu_val in [2.5, 3.0, 3.3, 3.5, 4.0]:
    p = {**init_params, "mu_league": jnp.array(float(mu_val))}
    ld, _ = log_density(model_cfb, (training_data,), {"N_teams": N_teams}, p)
    print(f"  mu_league={mu_val}  log_density={ld:.2f}")

# Sweep sp_weight with standardized sp_rating
print("\nsp_weight sweep (r=5, mu_league=3.3):")
for w_val in [-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0]:
    p = {**init_params, "sp_weight": jnp.array(float(w_val))}
    ld, _ = log_density(model_cfb, (training_data,), {"N_teams": N_teams}, p)
    print(f"  sp_weight={w_val:<5.1f}  log_density={ld:.2f}")

numpyro.enable_validation(True)

log_density at prior means (all betas=0, r=5) : -13241.22

r_negbinom sweep (all else at prior means):
  r=0.01    log_density=-25225.50
  r=0.10    log_density=-18702.32
  r=0.50    log_density=-15254.65
  r=1.00    log_density=-14209.65
  r=2.00    log_density=-13499.55
  r=5.00    log_density=-13241.22
  r=10.00   log_density=-13690.70
  r=20.00   log_density=-14666.99
  r=50.00   log_density=-16314.43

mu_league sweep (r=5, all betas=0):
  mu_league=2.5  log_density=-17471.92
  mu_league=3.0  log_density=-13594.29
  mu_league=3.3  log_density=-13241.22
  mu_league=3.5  log_density=-13668.98
  mu_league=4.0  log_density=-16561.64

sp_weight sweep (r=5, mu_league=3.3):
  sp_weight=-2.0   log_density=-46659.52
  sp_weight=-1.0   log_density=-22466.46
  sp_weight=-0.5   log_density=-15934.40
  sp_weight=0.0    log_density=-13241.22
  sp_weight=0.5    log_density=-13859.12
  sp_weight=1.0    log_density=-17533.31
  sp_weight=2.0    log_density=-33312.60


In [49]:
# =============================================================================
# FEATURE AUDIT — do not add to notebook yet
#
# For every feature entering model_cfb(), checks:
#   1. Raw scale (mean, std, min, max)
#   2. Distribution shape (sparse? skewed? categorical?)
#   3. How it enters log_mu (linear, masked, index)
#   4. Whether the prior is appropriate given the scale
#   5. Any encoding issues
# =============================================================================

import pandas as pd
import numpy as np

# Reload clean unscaled data for audit
cur.execute("SELECT * FROM int.int_game_model_features WHERE season IN (2022,2023,2024)")
rows = cur.fetchall()
audit_cols = [d[0] for d in cur.description]
df_audit = pd.DataFrame(rows, columns=audit_cols)
numeric_audit = [c for c in df_audit.columns if c not in ['team_name','opponent','conference']]
df_audit[numeric_audit] = df_audit[numeric_audit].astype(float)

print("=" * 90)
print(f"{'FEATURE':<40} {'MEAN':>8} {'STD':>8} {'MIN':>8} {'MAX':>8} {'ZEROS%':>7} {'NULLS':>6}")
print("=" * 90)

continuous_features = [
    # Prior seeds
    ('sp_rating',                    'continuous', 'linear',  'Normal(0,1.0)',  'team-level prior seed'),
    ('recruiting_3yr_avg',           'continuous', 'masked',  'Normal(0,0.5)',  'conf-indexed prior seed'),

    # Universal game-level
    ('close_game_epa_per_play',      'continuous', 'linear',  'Normal(0,0.5)',  'anchor'),
    ('close_game_def_epa_per_play',  'continuous', 'linear',  'Normal(0,0.5)',  'anchor'),
    ('pregame_elo',                  'continuous', 'linear',  'Normal(0,0.3)',  'game-level'),
    ('elo_sp_divergence',            'continuous', 'linear',  'Normal(0,0.2)',  'derived — already z-scored difference'),
    ('last3_win_pct',                'continuous', 'linear',  'Normal(0,0.3)',  'game-level momentum'),
    ('away_travel_distance_mi',      'sparse',     'linear',  'Normal(0,0.3)',  'threshold-zeroed <1500mi'),
    ('away_tz_delta_hrs',            'sparse',     'linear',  'Normal(0,0.3)',  'threshold-zeroed abs<2hr'),
    ('wind_chill',                   'sparse',     'linear',  'Normal(0,0.2)',  'threshold-zeroed >40F or dome'),
    ('rush_rate_std_downs_delta',    'continuous', 'linear',  'Normal(0,0.3)',  'style delta'),
    ('rush_rate_pass_downs_delta',   'continuous', 'linear',  'Normal(0,0.3)',  'style delta'),
    ('off_sack_rate_allowed_delta',  'continuous', 'linear',  'Normal(0,0.2)',  'sack mismatch'),
    ('def_sack_rate_delta',          'continuous', 'linear',  'Normal(0,0.2)',  'sack mismatch'),

    # Archetype matchups — CATEGORICAL
    ('offense_archetype_matchup',    'categorical','linear',  'Normal(0,0.3)',  'INTEGER CODE — not continuous'),
    ('defense_archetype_matchup',    'categorical','linear',  'Normal(0,0.3)',  'INTEGER CODE — not continuous'),
    ('home_off_vs_away_def_matchup', 'categorical','linear',  'Normal(0,0.3)',  'INTEGER CODE — not continuous'),
    ('away_off_vs_home_def_matchup', 'categorical','linear',  'Normal(0,0.3)',  'INTEGER CODE — not continuous'),

    # Conference-scoped
    ('last3_off_epa_avg',            'continuous', 'masked',  'Normal(0,0.3)',  'scoped: ACC/MAC/SEC'),
    ('last3_def_epa_avg',            'continuous', 'masked',  'Normal(0,0.3)',  'scoped: AmAth/B10/CUSA/MAC/P12/SB'),
    ('last3_points_scored_avg',      'continuous', 'masked',  'Normal(0,0.3)',  'scoped: ACC/B12/B10/CUSA/MAC/MWC'),
    ('last3_points_allowed_avg',     'continuous', 'masked',  'Normal(0,0.3)',  'scoped: AmAth/B10/CUSA/MWC/P12/SB'),
    ('days_since_last_game',         'sparse',     'masked',  'Normal(0,0.2)',  'scoped: AmAth/B12, threshold-zeroed <12d'),
    ('close_game_play_count_delta',  'continuous', 'masked',  'Normal(0,0.2)',  'scoped: ACC/AmAth/B12/MAC/P12/SB'),
    ('away_elevation_delta_ft',      'sparse',     'masked',  'Normal(0,0.3)',  'scoped: MWC/B12, threshold-zeroed <2000ft'),
]

issues = []

for col, ftype, entry, prior, notes in continuous_features:
    vals = df_audit[col].values
    mean = vals.mean()
    std  = vals.std()
    vmin = vals.min()
    vmax = vals.max()
    zero_pct = (vals == 0).mean() * 100
    nulls = pd.isna(vals).sum()

    print(f"{col:<40} {mean:>8.3f} {std:>8.3f} {vmin:>8.3f} {vmax:>8.3f} {zero_pct:>6.1f}% {nulls:>6}")

    # Flag issues
    if ftype == 'categorical':
        issues.append(f"CATEGORICAL ENCODING: {col} — integer codes fed as continuous. "
                      f"Categories: {sorted(df_audit[col].unique().tolist())}. "
                      f"Must be dropped or properly embedded before fit.")

    if ftype in ('continuous', 'masked') and std > 5:
        issues.append(f"SCALE: {col} std={std:.2f} — needs standardization. "
                      f"With prior {prior}, 1-SD move contributes ±{std:.1f} to log_mu.")

    if ftype == 'sparse':
        if zero_pct > 80:
            issues.append(f"SPARSE: {col} is {zero_pct:.1f}% zeros. Standardizing this "
                          f"will produce a non-normal distribution. Prior {prior} "
                          f"not well-calibrated to a 1-SD move on a sparse feature.")

    if col == 'elo_sp_divergence':
        issues.append(f"DOUBLE STANDARDIZATION RISK: {col} is already a difference of "
                      f"two z-scores (computed in model_03 Cell 7). Standardizing again "
                      f"in Cell 5 changes its interpretation. Confirm this is intentional.")

print()
print("=" * 90)
print(f"\nFEATURES AUDITED : {len(continuous_features)}")
print(f"ISSUES FOUND     : {len(issues)}")
print()
print("ISSUES:")
print("-" * 90)
for i, issue in enumerate(issues, 1):
    print(f"\n{i}. {issue}")

print()
print("-" * 90)
print("\nNON-FEATURE COLUMNS IN int.int_game_model_features (loaded but not in GameData):")
gamedata_cols = [col for col, *_ in continuous_features]
gamedata_cols += ['is_home', 'points_scored', 'game_id', 'season',
                  'team_name', 'opponent', 'conference']
unused = [c for c in df_audit.columns if c not in gamedata_cols]
for col in unused:
    vals = df_audit[col].values
    print(f"  {col:<35} mean={vals.mean():.3f}  std={vals.std():.3f}  "
          f"-- confirm intentionally excluded from model")

FEATURE                                      MEAN      STD      MIN      MAX  ZEROS%  NULLS
sp_rating                                   1.090   12.874  -33.000   35.300    0.0%      0
recruiting_3yr_avg                        175.943   54.826   18.690  326.050    0.0%      0
close_game_epa_per_play                     0.172    0.191   -0.519    1.018    0.2%      0
close_game_def_epa_per_play                 0.172    0.191   -0.519    1.018    0.2%      0
pregame_elo                              1511.452  236.138  805.000 2233.000    0.0%      0
elo_sp_divergence                          -0.000    0.448   -1.821    1.861    0.0%      0
last3_win_pct                               0.540    0.327    0.000    1.000   14.7%      0
away_travel_distance_mi                    56.956  365.081    0.000 4091.700   97.4%      0
away_tz_delta_hrs                           0.001    0.406   -3.000    3.000   97.4%      0
wind_chill                                  3.250    9.775    0.000   39.976   8